# langchain
- a framework that helps build applications using LLMs.
1. Chat Models: is a component/interface designed to communicate in a structured way with LLMs.
    1. consistent workflow
    2. easy switching between chat models
    3. context management
    4. efficient chaining
    5. scalability
2. Prompt templates: 
3. Chains: 
    1. extended or sequential chaining
    2. parallel chaining
    3. conditional chaining
4. RAGs:
5. Agents and tools:

## Chat models

https://python.langchain.com/docs/integrations/chat/

In [ ]:
# pip install -qU langchain-openai

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-3.5-turbo")

res = llm.invoke("What is the capital of France?")

res.content

In [ ]:
# system message: defines the AI role and sets the context for the conversation

from langchain_core.messages import AIMessage, SystemMessage, HumanMessage

messages = [
    SystemMessage("You are an expert in social media strategy."),
    HumanMessage("Give a short tip to create engaging posts on instagram.")
]

res = llm.invoke(messages)
res.content

In [ ]:
model = ChatOpenAI(model="gpt-3.5-turbo")
chat_history = []

system_prompt = SystemMessage(content="You are a helpful AI assistant.")
chat_history.append(system_prompt)

while True:
    query = input("You: ")
    if query.lower() == "quit":
        break
    chat_history.append(HumanMessage(content=query))
    response = model.invoke(chat_history)
    chat_history.append(AIMessage(content=response.content))
    print(f"AI: {response.content}")

## Prompt templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
llm = ChatOpenAI(model="gpt-3.5-turbo")

template = "Write a {tone} email to {company} expressing my interest in their {position} position."
prompt_template = ChatPromptTemplate.from_template(template=template)
prompt = prompt_template.invoke({
    "tone": "professional",
    "company": "Acme Corp",
    "position": "Software Engineer"
})
res = llm.invoke(prompt)
res.content

In [ ]:
messages = [
    ("system", "You are a helpful AI assistant who has extensive knowledge about {topic}."),
    ("human", "tell me {number} interesting facts.")
]
prompt_template = ChatPromptTemplate.from_messages(messages)

prompt = prompt_template.invoke({
    "topic": "Python programming",
    "number": 3
})

## Chains

In [ ]:
from langchain.schema.output_parser import StrOutputParser
model = ChatOpenAI(model="gpt-3.5-turbo")

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a facts expert who knows facts about {animal}"),
    ("human", "What are {fact_count}interesting facts.")
])


chain = prompt_template | model | StrOutputParser() # extracts content property

res = chain.invoke({"animal":"cat","fact_count": 3}) # whatever we pass here is going to be avalable across the chain


In [ ]:
# RunnableLambda is a simple wrapper that lets us create each task as a single reusable unit
from langchain.schema.runnables import RunnableLambda, RunnableSequence
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a facts expert who knows facts about {animal}"),
    ("human", "What are {fact_count}interesting facts.")
])
format_prompt = RunnableLambda(lambda x: prompt_template.format_prompt(**x)) # only replaces the plaehoder values 
invoke_model = RunnableLambda(lambda x: model.invoke(x.to_messages())) # takes a formatted prompt object and converts it into an explicit list of message objects (SystemMessage, HumanMessage, etc.) that a Chat Model expects to receive.
parse_output = RunnableLambda(lambda x: x.content)

chain = RunnableSequence(first=format_prompt, middle=[invoke_model], last=parse_output)
res = chain.invoke({"animal":"cat","fact_count": 3})